<a href="https://colab.research.google.com/github/danielabarba29-star/Analisis-hsd17b3_Bioinform-tica_gen-mica/blob/main/Pr%C3%A1ctica_6_Bioinform%C3%A1tica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Script que entra a las páginas de resultados del trabajo de docking en HDOCK Server siguiendo los hipervínculos de los primeros 10 modelos: model_1 ... model_10, parsea los tres bloques de cada modelo y deja todo en formato tidy (largo) que es justo lo que Power BI y pandas necesitan.

Para correrlo en Colab necesitamos las dependencias:

!pip -q install requests beautifulsoup4 lxml pandas openpyxl

Pega el contenido de hdock_scraper.py en una celda, edita solo el diccionario JOBS con las URLs de los dos jobs (la de la página de resultados, tipo http://hdock.phys.hust.edu.cn/data/<id>/) y ejecuta. Al final descargas los CSV con files.download().

Qué hace cada pieza
find_model_links() no asume el nombre de los archivos: lee el HTML de la página de resultados y recoge los href de las anclas cuyo texto sea model_1…model_10. Así funciona aunque HDOCK cambie la ruta interna. Si no encuentra anclas, guess_model_links() prueba los nombres habituales (model_1.txt, interface_1.txt, etc.).

parse_model_text() es el parser real y está deliberadamente separado de la descarga: recibe texto plano, detecta los tres encabezados # y aplica dos regex, una para líneas de residuo (TYR 164A 4.020) y otra para pares (164A - 48S 4.020). Tolera los tabuladores y las líneas “vacías con un espacio” que mete el servidor. Esto te sirve de red de seguridad: si algún job ya expiró, le pasas el texto pegado a mano y obtienes exactamente la misma estructura.
parse_summary_table() saca Docking Score / Confidence Score / Ligand rmsd de la tabla del top 10. Intenta primero pd.read_html y si la tabla no es un <table> real cae a parseo de texto.

Lo que obtienes
| Archivo | Contenido |
| --- | --- |
| `hdock_summary.csv` | job, model, docking_score, confidence_score, ligand_rmsd |
| `hdock_receptor_residues.csv` | Un renglón por residuo de receptor por modelo |
| `hdock_ligand_residues.csv` | Lo mismo para el ligando (nucleótidos DA/DG/DT/DC) |
| `hdock_pairs.csv` | **Tabla clave:** job, model, receptor_resid/chain, ligand_resid/chain, `pair_id` (ej. `164A-48S`), distance |
| `hdock_matriz_pares.csv` | Matriz pares × (job, modelo) con la distancia mínima |
| `hdock_pairs.xlsx` | Una hoja por modelo, replicando el layout manual |

In [ ]:
"""
hdock_scraper.py
================
Descarga y estructura los resultados de interfaz de uno o varios jobs del
servidor HDOCK (http://hdock.phys.hust.edu.cn/) sin copiar y pegar a mano.

Para cada job:
  1. Abre la página de resultados.
  2. Localiza los hipervínculos model_1 ... model_N.
  3. Descarga y parsea los tres bloques de cada modelo:
       - # Receptor interface residue(s):
       - # Ligand interface residue(s):
       - # Receptor-ligand interface residue pair(s):
  4. Parsea la tabla "Summary of the Top 10 Models"
     (Docking Score / Confidence Score / Ligand rmsd).

Salidas (formato largo, listo para pandas y Power BI):
  hdock_summary.csv            job, model, docking_score, confidence_score, ligand_rmsd
  hdock_receptor_residues.csv  job, model, resname, resid, chain, distance
  hdock_ligand_residues.csv    job, model, resname, resid, chain, distance
  hdock_pairs.csv              job, model, receptor_*, ligand_*, pair_id, distance
  hdock_pairs.xlsx (opcional)  una hoja por modelo, replicando el layout manual

Uso en Google Colab:
  !pip -q install requests beautifulsoup4 lxml pandas openpyxl
  (pegar este archivo en una celda, editar el diccionario JOBS y ejecutar)

Si el servidor ya borró el job, usa parse_model_text() con el texto pegado
a mano: el parser es el mismo.
"""

'\nhdock_scraper.py\n================\nDescarga y estructura los resultados de interfaz de uno o varios jobs del\nservidor HDOCK (http://hdock.phys.hust.edu.cn/) sin copiar y pegar a mano.\n \nPara cada job:\n  1. Abre la página de resultados.\n  2. Localiza los hipervínculos model_1 ... model_N.\n  3. Descarga y parsea los tres bloques de cada modelo:\n       - # Receptor interface residue(s):\n       - # Ligand interface residue(s):\n       - # Receptor-ligand interface residue pair(s):\n  4. Parsea la tabla "Summary of the Top 10 Models"\n     (Docking Score / Confidence Score / Ligand rmsd).\n \nSalidas (formato largo, listo para pandas y Power BI):\n  hdock_summary.csv            job, model, docking_score, confidence_score, ligand_rmsd\n  hdock_receptor_residues.csv  job, model, resname, resid, chain, distance\n  hdock_ligand_residues.csv    job, model, resname, resid, chain, distance\n  hdock_pairs.csv              job, model, receptor_*, ligand_*, pair_id, distance\n  hdock_pair

In [ ]:
!pip -q install requests beautifulsoup4 lxml pandas openpyxl

In [ ]:
from __future__ import annotations

import io
import re
import time
import urllib.parse
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

In [ ]:
# ---------------------------------------------------------------------------
# 1. CONFIGURACIÓN  --  edita solo esta sección
# ---------------------------------------------------------------------------

# Pega la URL de la página de resultados de cada job de HDOCK.
# Es la URL que te llegó por correo o que tienes abierta en el navegador,
# del estilo: http://hdock.phys.hust.edu.cn/data/<id_del_job>/

# Ejemplo general:
# JOBS = {
#    "nativa": "http://hdock.phys.hust.edu.cn/data/PEGA_AQUI_EL_ID_NATIVA/",
#    "mutada": "http://hdock.phys.hust.edu.cn/data/PEGA_AQUI_EL_ID_MUTADA/",
#}


JOBS = {
#    "nativa": "http://hdock.phys.hust.edu.cn/data/6ae9fcb02f485/",
#   "mutada": "http://hdock.phys.hust.edu.cn/data/6aea39e305b2d/",
    "nativa": "http://hdock.phys.hust.edu.cn/data/6ae9fd0e40d18/",
    "mutada": "http://hdock.phys.hust.edu.cn/data/6ae9fd474a999/",
    }

OUTDIR = Path(".")          # carpeta de salida
MAX_MODELS = 10             # cuántos modelos del top bajar
PAUSE = 1.0                 # segundos entre peticiones (sé amable con el servidor)
WRITE_EXCEL = True          # también generar el .xlsx con una hoja por modelo
TIMEOUT = 30

HEADERS = {"User-Agent": "Mozilla/5.0 (academic use; HDOCK result parser)"}

In [ ]:
# ---------------------------------------------------------------------------
# 2. PARSERS  --  funcionan igual con texto descargado o pegado a mano
# ---------------------------------------------------------------------------

MODEL_RE = re.compile(r"=+\s*Model\s+(\d+)\s*=+", re.IGNORECASE)

SECTIONS = {
    "receptor": re.compile(r"#\s*Receptor\s+interface\s+residue", re.IGNORECASE),
    "ligand": re.compile(r"#\s*Ligand\s+interface\s+residue", re.IGNORECASE),
    "pairs": re.compile(r"#\s*Receptor-ligand\s+interface\s+residue\s+pair", re.IGNORECASE),
}

# "TYR    164A     4.020"  ->  resname, resid, chain, distancia
RES_RE = re.compile(r"^\s*([A-Za-z]{1,3})\s+(\d+)\s*([A-Za-z])\s+([-\d.]+)\s*$")

# "164A  -  48S     4.020"  ->  resid_rec, chain_rec, resid_lig, chain_lig, distancia
PAIR_RE = re.compile(
    r"^\s*(\d+)\s*([A-Za-z])\s*-\s*(\d+)\s*([A-Za-z])\s+([-\d.]+)\s*$"
)

# Líneas sueltas del pie del archivo, por si vienen pegadas al texto
SCORE_RE = re.compile(r"Docking\s+Score\s*[:\t ]+\s*([-\d.]+)", re.IGNORECASE)
CONF_RE = re.compile(r"Confidence\s+Score\s*[:\t ]+\s*([-\d.]+)", re.IGNORECASE)
RMSD_RE = re.compile(r"Ligand\s+rmsd[^\d-]*([-\d.]+)", re.IGNORECASE)


def parse_model_text(text: str, job: str = "", model: int | None = None) -> dict:
    """Parsea el bloque de texto de UN modelo de HDOCK.

    Devuelve un dict con las llaves 'receptor', 'ligand', 'pairs' (listas de
    dicts) y 'scores' (dict). Tolera tabuladores, espacios múltiples y las
    líneas en blanco con un espacio que mete el servidor.
    """
    if model is None:
        m = MODEL_RE.search(text)
        model = int(m.group(1)) if m else 0

    receptor, ligand, pairs = [], [], []
    current = None

    for raw in text.splitlines():
        line = raw.replace("\t", "    ").rstrip()
        if not line.strip():
            continue

        # ¿Es un encabezado de sección?
        matched_header = False
        for name, rx in SECTIONS.items():
            if rx.search(line):
                current = name
                matched_header = True
                break
        if matched_header or MODEL_RE.search(line):
            continue

        if current == "pairs":
            m = PAIR_RE.match(line)
            if m:
                rid, rch, lid, lch, dist = m.groups()
                pairs.append(
                    {
                        "job": job,
                        "model": model,
                        "receptor_resid": int(rid),
                        "receptor_chain": rch.upper(),
                        "ligand_resid": int(lid),
                        "ligand_chain": lch.upper(),
                        "pair_id": f"{rid}{rch.upper()}-{lid}{lch.upper()}",
                        "distance": float(dist),
                    }
                )
        elif current in ("receptor", "ligand"):
            m = RES_RE.match(line)
            if m:
                resname, resid, chain, dist = m.groups()
                row = {
                    "job": job,
                    "model": model,
                    "resname": resname.upper(),
                    "resid": int(resid),
                    "chain": chain.upper(),
                    "residue_id": f"{resid}{chain.upper()}",
                    "label": f"{resname.upper()}{resid}{chain.upper()}",
                    "distance": float(dist),
                }
                (receptor if current == "receptor" else ligand).append(row)

    scores = {}
    for key, rx in (("docking_score", SCORE_RE),
                    ("confidence_score", CONF_RE),
                    ("ligand_rmsd", RMSD_RE)):
        m = rx.search(text)
        if m:
            scores[key] = float(m.group(1))

    return {"model": model, "receptor": receptor, "ligand": ligand,
            "pairs": pairs, "scores": scores}


def parse_summary_table(html: str, job: str) -> pd.DataFrame:
    """Extrae la tabla 'Summary of the Top 10 Models' de la página del job."""
    rows = {}

    # Intento 1: lectura directa de las tablas HTML
    try:
        for df in pd.read_html(io.StringIO(html)):
            flat = df.astype(str).values.ravel()
            if any("Docking Score" in str(v) for v in flat):
                dfi = df.set_index(df.columns[0])
                for key, name in (("Docking Score", "docking_score"),
                                  ("Confidence Score", "confidence_score"),
                                  ("Ligand rmsd", "ligand_rmsd")):
                    for idx in dfi.index:
                        if key.lower() in str(idx).lower():
                            vals = pd.to_numeric(
                                dfi.loc[idx], errors="coerce"
                            ).dropna().tolist()
                            rows[name] = vals
                            break
                if rows:
                    break
    except Exception:
        pass

    # Intento 2: sobre el texto plano, por si la tabla no es <table>
    if not rows:
        text = BeautifulSoup(html, "html.parser").get_text("\n")
        for line in text.splitlines():
            low = line.lower()
            nums = re.findall(r"-?\d+\.\d+", line)
            if "docking score" in low and len(nums) > 1:
                rows["docking_score"] = [float(x) for x in nums]
            elif "confidence score" in low and len(nums) > 1:
                rows["confidence_score"] = [float(x) for x in nums]
            elif "ligand rmsd" in low and len(nums) > 1:
                rows["ligand_rmsd"] = [float(x) for x in nums]

    if not rows:
        return pd.DataFrame()

    n = min(len(v) for v in rows.values())
    out = pd.DataFrame({k: v[:n] for k, v in rows.items()})
    out.insert(0, "model", range(1, n + 1))
    out.insert(0, "job", job)
    return out

In [ ]:
# ---------------------------------------------------------------------------
# 3. DESCARGA
# ---------------------------------------------------------------------------

def normalize_base(url: str) -> str:
    """Garantiza la diagonal final: sin ella, urljoin() se come el ID del job."""
    url = url.strip()
    return url if url.endswith("/") else url + "/"

def find_model_links(html: str, base_url: str, max_models: int) -> dict[int, str]:
    """Localiza los hipervínculos model_1 ... model_N en la página del job."""
    soup = BeautifulSoup(html, "html.parser")
    links: dict[int, str] = {}

    for a in soup.find_all("a", href=True):
        txt = a.get_text(strip=True)
        m = re.fullmatch(r"model[_\s-]?(\d+)", txt, re.IGNORECASE)
        if not m:
            m = re.search(r"model[_\s-]?(\d+)", a["href"], re.IGNORECASE)
        if m:
            idx = int(m.group(1))
            if 1 <= idx <= max_models:
                links[idx] = urllib.parse.urljoin(base_url, a["href"])

    return dict(sorted(links.items()))


def guess_model_links(base_url: str, session: requests.Session,
                      max_models: int) -> dict[int, str]:
    """Plan B: prueba nombres de archivo habituales si no hubo anclas."""
    templates = [
        "interface_{i}.txt",           # <-- MOVER/AGREGAR AL INICIO
        "model_{i}.txt", "model{i}.txt", # "interface_{i}.txt", # Posición original
        "model_{i}.html", "model_{i}.pdb.txt",
    ]
    for tpl in templates:
        url = urllib.parse.urljoin(base_url, tpl.format(i=1))
        try:
            r = session.get(url, headers=HEADERS, timeout=TIMEOUT)
            if r.ok and MODEL_RE.search(r.text):
                return {i: urllib.parse.urljoin(base_url, tpl.format(i=i))
                        for i in range(1, max_models + 1)}
        except requests.RequestException:
            continue
    return {}


def fetch_text(url: str, session: requests.Session) -> str:
    r = session.get(url, headers=HEADERS, timeout=TIMEOUT)
    r.raise_for_status()
    ctype = r.headers.get("Content-Type", "")
    if "html" in ctype.lower() or r.text.lstrip().lower().startswith("<"):
        return BeautifulSoup(r.text, "html.parser").get_text("\n")
    return r.text


def scrape_job(job: str, url: str, session: requests.Session) -> dict:
    url = normalize_base(url)          # <-- AGREGAR ESTA LÍNEA
    print(f"\n=== Job '{job}' ===")
    print(f"  Página de resultados: {url}")
    page = session.get(url, headers=HEADERS, timeout=TIMEOUT)
    page.raise_for_status()
    html = page.text

    summary = parse_summary_table(html, job)
    print(f"  Tabla resumen: {len(summary)} modelos"
          if not summary.empty else "  Tabla resumen: no encontrada")

    links = find_model_links(html, url, MAX_MODELS)
    if not links:
        print("  No se hallaron anclas model_N; probando nombres de archivo...")
        links = guess_model_links(url, session, MAX_MODELS)
    if not links:
        raise RuntimeError(
            f"No pude localizar los enlaces de los modelos en {url}. "
            "Abre la página, copia la URL de 'model_1' y pásala a "
            "fetch_text() manualmente, o usa parse_model_text() con el "
            "texto pegado."
        )
    print(f"  Modelos localizados: {sorted(links)}")

    receptor, ligand, pairs, scores = [], [], [], []
    for idx, link in links.items():
        try:
            text = fetch_text(link, session)
        except requests.RequestException as exc:
            print(f"  ! Modelo {idx}: fallo de descarga ({exc})")
            continue
        parsed = parse_model_text(text, job=job, model=idx)
        receptor += parsed["receptor"]
        ligand += parsed["ligand"]
        pairs += parsed["pairs"]
        if parsed["scores"]:
            scores.append({"job": job, "model": idx, **parsed["scores"]})
        print(f"  Modelo {idx:>2}: {len(parsed['receptor']):>3} res. receptor | "
              f"{len(parsed['ligand']):>3} res. ligando | "
              f"{len(parsed['pairs']):>3} pares")
        time.sleep(PAUSE)

    # Si la tabla resumen falló, usamos los scores del pie de cada modelo
    if summary.empty and scores:
        summary = pd.DataFrame(scores)

    return {
        "summary": summary,
        "receptor": pd.DataFrame(receptor),
        "ligand": pd.DataFrame(ligand),
        "pairs": pd.DataFrame(pairs),
    }

In [ ]:
# ---------------------------------------------------------------------------
# 4. ESCRITURA
# ---------------------------------------------------------------------------

def write_excel_per_model(pairs: pd.DataFrame, receptor: pd.DataFrame,
                          ligand: pd.DataFrame, path: Path) -> None:
    """Genera un .xlsx con una hoja por (job, modelo), como el armado manual."""
    with pd.ExcelWriter(path, engine="openpyxl") as xl:
        keys = pairs[["job", "model"]].drop_duplicates().itertuples(index=False)
        for job, model in keys:
            sheet = f"{job}_m{model}"[:31]
            blocks = []
            for title, df, cols in (
                ("# Receptor interface residue(s):", receptor,
                 ["resname", "residue_id", "distance"]),
                ("# Ligand interface residue(s):", ligand,
                 ["resname", "residue_id", "distance"]),
            ):
                sub = df[(df.job == job) & (df.model == model)][cols]
                blocks.append(pd.DataFrame({cols[0]: [title]}))
                blocks.append(sub)
                blocks.append(pd.DataFrame({cols[0]: [""]}))

            sub = pairs[(pairs.job == job) & (pairs.model == model)]
            head = pd.DataFrame({"resname": ["# Receptor-ligand interface residue pair(s):"]})
            body = pd.DataFrame({
                "resname": sub["pair_id"].values,
                "residue_id": "",
                "distance": sub["distance"].values,
            })
            blocks += [head, body]

            pd.concat(blocks, ignore_index=True).to_excel(
                xl, sheet_name=sheet, index=False
            )
    print(f"  -> {path}")


def main() -> None:
    OUTDIR.mkdir(parents=True, exist_ok=True)
    session = requests.Session()

    all_parts = {"summary": [], "receptor": [], "ligand": [], "pairs": []}
    for job, url in JOBS.items():
        if "PEGA_AQUI" in url:
            print(f"! Job '{job}': falta pegar la URL real. Lo omito.")
            continue
        try:
            res = scrape_job(job, url, session)
        except Exception as exc:
            print(f"! Job '{job}' falló: {exc}")
            continue
        for key, df in res.items():
            if not df.empty:
                all_parts[key].append(df)

    print("\n=== Archivos generados ===")
    frames = {}
    for key, parts in all_parts.items():
        if not parts:
            continue
        df = pd.concat(parts, ignore_index=True)
        frames[key] = df
        out = OUTDIR / f"hdock_{'summary' if key == 'summary' else key + ('_residues' if key in ('receptor', 'ligand') else '')}.csv"
        df.to_csv(out, index=False, encoding="utf-8-sig")
        print(f"  -> {out}  ({len(df)} filas)")

    if WRITE_EXCEL and "pairs" in frames:
        write_excel_per_model(
            frames["pairs"],
            frames.get("receptor", pd.DataFrame()),
            frames.get("ligand", pd.DataFrame()),
            OUTDIR / "hdock_pairs.xlsx",
        )

    # Matriz pares x job, útil como punto de partida del análisis de pares comunes
    if "pairs" in frames:
        matriz = (
            frames["pairs"]
            .pivot_table(index="pair_id", columns=["job", "model"],
                         values="distance", aggfunc="min")
        )
        out = OUTDIR / "hdock_matriz_pares.csv"
        matriz.to_csv(out, encoding="utf-8-sig")
        print(f"  -> {out}  ({matriz.shape[0]} pares únicos x {matriz.shape[1]} modelos)")

In [ ]:
if __name__ == "__main__":
    main()


=== Job 'nativa' ===
  Página de resultados: http://hdock.phys.hust.edu.cn/data/6ae9fd0e40d18/
  Tabla resumen: 10 modelos
  Modelos localizados: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
  Modelo  1:  15 res. receptor |  12 res. ligando |  31 pares
  Modelo  2:  23 res. receptor |  23 res. ligando |  54 pares
  Modelo  3:  15 res. receptor |  13 res. ligando |  33 pares
  Modelo  4:  18 res. receptor |  15 res. ligando |  39 pares
  Modelo  5:  14 res. receptor |  13 res. ligando |  31 pares
  Modelo  6:  17 res. receptor |  15 res. ligando |  37 pares
  Modelo  7:  14 res. receptor |  10 res. ligando |  28 pares
  Modelo  8:  22 res. receptor |  17 res. ligando |  49 pares
  Modelo  9:  22 res. receptor |  21 res. ligando |  51 pares
  Modelo 10:  19 res. receptor |  15 res. ligando |  37 pares

=== Job 'mutada' ===
  Página de resultados: http://hdock.phys.hust.edu.cn/data/6ae9fd474a999/
  Tabla resumen: 10 modelos
  Modelos localizados: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
  Modelo  1:  26 res. 

# Visualizaciones

In [ ]:
!pip -q install matplotlib-venn networkx

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

matplotlib.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

FIGDIR = Path("figuras")
TABDIR = Path("tablas")

# Modelo sobre el que se hacen las figuras de detalle (1 = mejor score)
MODELO_FOCO = 1
# Modelos a considerar en el análisis de pares comunes
MODELOS_COMPARAR = [1]

# --- Clasificación química -------------------------------------------------

NUCLEOTIDOS = {"DA", "DT", "DG", "DC", "DU", "A", "T", "G", "C", "U",
               "ADE", "THY", "GUA", "CYT", "URA"}

CLASES_AA = {
    "Básico (+)":    {"ARG", "LYS", "HIS"},
    "Ácido (−)":     {"ASP", "GLU"},
    "Polar":         {"SER", "THR", "ASN", "GLN", "CYS"},
    "Aromático":     {"PHE", "TRP", "TYR"},
    "Hidrofóbico":   {"ALA", "VAL", "LEU", "ILE", "MET", "PRO", "GLY"},
}
COLOR_CLASE = {
    "Básico (+)": "#2166AC", "Ácido (−)": "#B2182B", "Polar": "#4DAF4A",
    "Aromático": "#984EA3", "Hidrofóbico": "#999999", "Otro": "#DDDDDD",
}
COLOR_JOB = {"nativa": "#2166AC", "mutada": "#B2182B"}

# Umbrales de distancia (Å) — ver pregunta 4 del pre-reporte
UMBRAL_CHOQUE, UMBRAL_HBOND, UMBRAL_VDW = 2.2, 3.5, 5.0


def clase_quimica(resname: str) -> str:
    for clase, miembros in CLASES_AA.items():
        if resname in miembros:
            return clase
    return "Otro"


def es_nucleotido(resname) -> bool:
    return str(resname).strip().upper() in NUCLEOTIDOS

In [ ]:
# ---------------------------------------------------------------------------
# 1. CARGA Y NORMALIZACIÓN
# ---------------------------------------------------------------------------

def cargar_y_normalizar() -> pd.DataFrame:
    """Devuelve la tabla de pares con proteína y ADN SIEMPRE en las mismas
    columnas, sin importar cuál fue receptor y cuál ligando en cada job.

    Columnas de salida:
        job, model, aa_name, aa_resid, aa_chain, aa_label,
        nt_name, nt_resid, nt_chain, nt_label, distance, pair_key
    """
    pares = pd.read_csv("hdock_pairs.csv")
    rec = pd.read_csv("hdock_receptor_residues.csv")
    lig = pd.read_csv("hdock_ligand_residues.csv")

    # El bloque de pares no trae el nombre del residuo: se recupera cruzando
    # con los bloques de residuos por (job, modelo, número, cadena).
    rec_k = rec[["job", "model", "resid", "chain", "resname"]].drop_duplicates()
    lig_k = lig[["job", "model", "resid", "chain", "resname"]].drop_duplicates()

    df = pares.merge(
        rec_k.rename(columns={"resid": "receptor_resid", "chain": "receptor_chain",
                              "resname": "receptor_name"}),
        on=["job", "model", "receptor_resid", "receptor_chain"], how="left",
    ).merge(
        lig_k.rename(columns={"resid": "ligand_resid", "chain": "ligand_chain",
                              "resname": "ligand_name"}),
        on=["job", "model", "ligand_resid", "ligand_chain"], how="left",
    )

    faltan = df[["receptor_name", "ligand_name"]].isna().any(axis=1).sum()
    if faltan:
        print(f"  ! {faltan} pares sin nombre de residuo (se descartan)")
        df = df.dropna(subset=["receptor_name", "ligand_name"])

    # ¿De qué lado quedó la proteína en cada renglón?
    rec_es_adn = df["receptor_name"].map(es_nucleotido)

    def escoge(col_rec, col_lig, cuando_rec_es_adn_tomar_ligando: bool):
        """Si el receptor es ADN, la proteína está del lado del ligando."""
        a, b = df[col_rec], df[col_lig]
        return np.where(rec_es_adn, b if cuando_rec_es_adn_tomar_ligando else a,
                        a if cuando_rec_es_adn_tomar_ligando else b)

    out = pd.DataFrame({
        "job": df["job"],
        "model": df["model"],
        # Proteína: lado del receptor salvo que el receptor sea ADN
        "aa_name":  escoge("receptor_name",  "ligand_name",  True),
        "aa_resid": escoge("receptor_resid", "ligand_resid", True),
        "aa_chain": escoge("receptor_chain", "ligand_chain", True),
        # ADN: el lado contrario
        "nt_name":  escoge("receptor_name",  "ligand_name",  False),
        "nt_resid": escoge("receptor_resid", "ligand_resid", False),
        "nt_chain": escoge("receptor_chain", "ligand_chain", False),
        "distance": df["distance"],
    })

    out["aa_resid"] = out["aa_resid"].astype(int)
    out["nt_resid"] = out["nt_resid"].astype(int)
    out["aa_label"] = out["aa_name"] + out["aa_resid"].astype(str)
    out["nt_label"] = out["nt_name"] + out["nt_resid"].astype(str) + out["nt_chain"]
    out["clase"] = out["aa_name"].map(clase_quimica)

    # Clave comparable entre jobs. La cadena de la PROTEÍNA se omite a
    # propósito: en un job es 'A' y en el otro 'L', y es la misma proteína.
    out["pair_key"] = (out["aa_resid"].astype(str) + "|"
                       + out["nt_chain"] + out["nt_resid"].astype(str))

    # Diagnóstico del orden original por job
    print("\n  Orden original de entrada en cada job:")
    for job, sub in df.groupby("job"):
        quien = "ADN" if es_nucleotido(sub["receptor_name"].iloc[0]) else "proteína"
        print(f"    {job:<8} -> receptor = {quien}"
              + ("   (invertido, se normalizó)" if quien == "ADN" else ""))

    print("\n  Cadenas detectadas tras normalizar:")
    for job, sub in out.groupby("job"):
        print(f"    {job:<8} proteína: {sorted(sub.aa_chain.unique())}  "
              f"ADN: {sorted(sub.nt_chain.unique())}  "
              f"ADN pos. {sub.nt_resid.min()}–{sub.nt_resid.max()}")

    return out

In [ ]:
# ---------------------------------------------------------------------------
# 2. FIGURAS
# ---------------------------------------------------------------------------

def fig1_mapa_contactos(df: pd.DataFrame) -> None:
    """Dato crudo: cada punto es un renglón de la tabla de pares."""
    jobs = sorted(df.job.unique())
    fig, axes = plt.subplots(1, len(jobs), figsize=(6 * len(jobs), 5.2))
    axes = np.atleast_1d(axes)

    for ax, job in zip(axes, jobs):
        sub = df[(df.job == job) & (df.model == MODELO_FOCO)]
        for chain, g in sub.groupby("nt_chain"):
            sc = ax.scatter(g.nt_resid, g.aa_resid,
                            s=(UMBRAL_VDW - g.distance.clip(upper=UMBRAL_VDW)) * 90 + 20,
                            c=g.distance, cmap="viridis_r",
                            vmin=1.5, vmax=UMBRAL_VDW,
                            edgecolor="white", linewidth=0.6,
                            marker="o" if chain == sub.nt_chain.iloc[0] else "s",
                            label=f"Cadena ADN {chain}")
        ax.set_xlabel("Posición en el ADN")
        ax.set_ylabel("Residuo de la proteína")
        ax.set_title(f"{job.capitalize()} — modelo {MODELO_FOCO}\n"
                     f"{len(sub)} contactos")
        ax.legend(frameon=False, fontsize=8)
        ax.grid(alpha=0.25, linestyle=":")

    cbar = fig.colorbar(sc, ax=axes.tolist(), shrink=0.85, pad=0.02)
    cbar.set_label("Distancia (Å) — puntos grandes y oscuros = contacto más cercano")
    fig.suptitle("Mapa de contactos: quién toca a quién", y=1.02, fontsize=13,
                 fontweight="bold")
    fig.savefig(FIGDIR / "fig1_contactos_modelo1.png")
    plt.close(fig)


def fig2_huella(df: pd.DataFrame) -> None:
    """¿Qué tramo del ADN queda cubierto, y cambia con la mutación?"""
    cob = (df.groupby(["job", "nt_chain", "nt_resid"])["model"]
             .nunique().reset_index(name="n_modelos"))
    chains = sorted(cob.nt_chain.unique())

    fig, axes = plt.subplots(len(chains), 1, figsize=(11, 3.1 * len(chains)),
                             squeeze=False)
    for ax, chain in zip(axes[:, 0], chains):
        sub = cob[cob.nt_chain == chain]
        jobs = sorted(sub.job.unique())
        ancho = 0.4
        for i, job in enumerate(jobs):
            g = sub[sub.job == job]
            ax.bar(g.nt_resid + (i - (len(jobs) - 1) / 2) * ancho, g.n_modelos,
                   width=ancho, label=job.capitalize(),
                   color=COLOR_JOB.get(job, "#777777"), alpha=0.9)
        ax.set_title(f"Cadena de ADN {chain}")
        ax.set_xlabel("Posición del nucleótido")
        ax.set_ylabel("Modelos en\ncontacto (de 10)")
        ax.legend(frameon=False)
        ax.grid(axis="y", alpha=0.25, linestyle=":")

    fig.suptitle("Huella sobre el ADN: ¿dónde se posa la proteína?",
                 y=1.0, fontsize=13, fontweight="bold")
    fig.tight_layout()
    fig.savefig(FIGDIR / "fig2_huella_adn.png")
    plt.close(fig)


def fig3_distancias(df: pd.DataFrame) -> None:
    """Conecta el dato con los criterios de la pregunta 4 del pre-reporte."""
    fig, ax = plt.subplots(figsize=(9.5, 5))
    bins = np.arange(1.4, 5.3, 0.1)

    for job in sorted(df.job.unique()):
        ax.hist(df[df.job == job].distance, bins=bins, alpha=0.55,
                label=job.capitalize(), color=COLOR_JOB.get(job, "#777777"))

    bandas = [
        (1.4, UMBRAL_CHOQUE, "#F4A582", "Choque estérico\n(imposible)"),
        (UMBRAL_CHOQUE, UMBRAL_HBOND, "#92C5DE", "Puente de H /\npuente salino"),
        (UMBRAL_HBOND, UMBRAL_VDW, "#D9D9D9", "van der Waals /\nhidrofóbico"),
    ]
    ymax = ax.get_ylim()[1]
    for x0, x1, color, texto in bandas:
        ax.axvspan(x0, x1, color=color, alpha=0.30, zorder=0)
        ax.text((x0 + x1) / 2, ymax * 0.93, texto, ha="center", va="top",
                fontsize=8.5, style="italic")
    for x in (UMBRAL_CHOQUE, UMBRAL_HBOND):
        ax.axvline(x, color="black", lw=1, ls="--", alpha=0.6)

    ax.set_xlabel("Distancia mínima entre átomos pesados (Å)")
    ax.set_ylabel("Número de contactos")
    ax.set_title("Distribución de distancias y tipo de interacción esperada")
    ax.legend(frameon=False, loc="upper left")
    fig.savefig(FIGDIR / "fig3_distancias.png")
    plt.close(fig)


def fig4_grado_aminoacidos(df: pd.DataFrame) -> None:
    """Grado de cada aminoácido en la red, coloreado por química."""
    jobs = sorted(df.job.unique())
    fig, axes = plt.subplots(1, len(jobs), figsize=(6.2 * len(jobs), 7),
                             squeeze=False)

    for ax, job in zip(axes[0], jobs):
        sub = df[(df.job == job) & (df.model == MODELO_FOCO)]
        g = (sub.groupby(["aa_label", "clase"]).size()
                .reset_index(name="n").sort_values("n"))
        ax.barh(g.aa_label, g.n,
                color=[COLOR_CLASE[c] for c in g.clase], edgecolor="white")
        ax.set_xlabel("Nucleótidos contactados (grado)")
        ax.set_title(f"{job.capitalize()} — modelo {MODELO_FOCO}")
        ax.grid(axis="x", alpha=0.25, linestyle=":")

    manijas = [plt.Rectangle((0, 0), 1, 1, color=c) for c in COLOR_CLASE.values()]
    fig.legend(manijas, COLOR_CLASE.keys(), ncol=len(COLOR_CLASE),
               frameon=False, loc="lower center", bbox_to_anchor=(0.5, -0.04))
    fig.suptitle("¿Qué aminoácidos sostienen la interfaz y de qué tipo son?",
                 y=1.01, fontsize=13, fontweight="bold")
    fig.tight_layout()
    fig.savefig(FIGDIR / "fig4_grado_aminoacidos.png")
    plt.close(fig)


def fig5_heatmap(df: pd.DataFrame) -> None:
    """La matriz bipartita completa de un modelo."""
    jobs = sorted(df.job.unique())
    fig, axes = plt.subplots(1, len(jobs), figsize=(7.5 * len(jobs), 6),
                             squeeze=False)

    for ax, job in zip(axes[0], jobs):
        sub = df[(df.job == job) & (df.model == MODELO_FOCO)]
        piv = sub.pivot_table(index="aa_label", columns="nt_label",
                              values="distance", aggfunc="min")
        piv = piv.reindex(index=sorted(piv.index, key=lambda s: int(s[3:])))
        im = ax.imshow(piv.values, cmap="viridis_r", aspect="auto",
                       vmin=1.5, vmax=UMBRAL_VDW)
        ax.set_xticks(range(len(piv.columns)))
        ax.set_xticklabels(piv.columns, rotation=90, fontsize=7.5)
        ax.set_yticks(range(len(piv.index)))
        ax.set_yticklabels(piv.index, fontsize=7.5)
        ax.set_title(f"{job.capitalize()} — modelo {MODELO_FOCO}")
        for i in range(len(piv.index)):
            for j in range(len(piv.columns)):
                v = piv.values[i, j]
                if not np.isnan(v):
                    ax.text(j, i, f"{v:.1f}", ha="center", va="center",
                            fontsize=5.5,
                            color="white" if v < 3.3 else "black")

    cbar = fig.colorbar(im, ax=axes[0].tolist(), shrink=0.8, pad=0.02)
    cbar.set_label("Distancia (Å)")
    fig.suptitle("Matriz de contactos aminoácido × nucleótido",
                 y=1.02, fontsize=13, fontweight="bold")
    fig.savefig(FIGDIR / "fig5_heatmap.png")
    plt.close(fig)


def fig6_pares_comunes(df: pd.DataFrame) -> pd.DataFrame:
    """El objetivo central: conservados, perdidos y ganados."""
    sub = df[df.model.isin(MODELOS_COMPARAR)]
    conj = {job: set(g.pair_key) for job, g in sub.groupby("job")}
    if len(conj) < 2:
        print("  ! Se necesitan dos jobs para comparar pares comunes")
        return pd.DataFrame()

    nat = conj.get("nativa", set())
    mut = conj.get("mutada", set())
    comunes, perdidos, ganados = nat & mut, nat - mut, mut - nat

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.2),
                                   gridspec_kw={"width_ratios": [1, 1.25]})

    try:
        from matplotlib_venn import venn2
        v = venn2([nat, mut], set_labels=("Nativa", "Mutada"), ax=ax1,
                  set_colors=(COLOR_JOB["nativa"], COLOR_JOB["mutada"]), alpha=0.55)
        for t in v.set_labels or []:
            if t: t.set_fontsize(11)
    except ImportError:
        ax1.bar(["Solo nativa\n(perdidos)", "Comunes\n(conservados)",
                 "Solo mutada\n(ganados)"],
                [len(perdidos), len(comunes), len(ganados)],
                color=[COLOR_JOB["nativa"], "#4DAF4A", COLOR_JOB["mutada"]])
        ax1.set_ylabel("Número de pares")
    ax1.set_title(f"Pares de contacto — modelo(s) {MODELOS_COMPARAR}")

    filas = []
    for clave, estado in ([(k, "Conservado") for k in comunes]
                          + [(k, "Perdido") for k in perdidos]
                          + [(k, "Ganado") for k in ganados]):
        filas.append({"pair_key": clave, "estado": estado})
    tabla = pd.DataFrame(filas)

    detalle = (sub.drop_duplicates(subset=["job", "pair_key"])
                  .merge(tabla, on="pair_key", how="left"))

    if not comunes:
        ax2.text(0.5, 0.5,
                 "No hay pares conservados\nentre nativa y mutada\n\n"
                 "Revisar si la numeración del ADN\nes equivalente entre jobs",
                 ha="center", va="center", fontsize=11, style="italic",
                 transform=ax2.transAxes)
        ax2.set_axis_off()
    else:
        d = detalle[detalle.estado == "Conservado"].copy()
        piv = d.pivot_table(index="pair_key", columns="job",
                            values="distance", aggfunc="min").dropna()
        piv = piv.sort_values(piv.columns[0])
        y = np.arange(len(piv))
        for job in piv.columns:
            ax2.scatter(piv[job], y, s=55, label=job.capitalize(),
                        color=COLOR_JOB.get(job, "#777"), zorder=3)
        ax2.hlines(y, piv.min(axis=1), piv.max(axis=1),
                   color="#BBBBBB", lw=1.5, zorder=1)
        ax2.set_yticks(y)
        ax2.set_yticklabels(piv.index, fontsize=7.5)
        ax2.axvline(UMBRAL_HBOND, color="black", ls="--", lw=1, alpha=0.5)
        ax2.set_xlabel("Distancia (Å)")
        ax2.set_title("Contactos conservados: ¿se acercan o se alejan?")
        ax2.legend(frameon=False)
        ax2.grid(axis="x", alpha=0.25, linestyle=":")

    fig.suptitle("Comparación de pares comunes: nativa vs. mutada",
                 y=1.02, fontsize=13, fontweight="bold")
    fig.tight_layout()
    fig.savefig(FIGDIR / "fig6_pares_comunes.png")
    plt.close(fig)

    detalle.to_csv(TABDIR / "tabla_pares_comunes.csv", index=False,
                   encoding="utf-8-sig")
    print(f"  Conservados: {len(comunes)} | Perdidos: {len(perdidos)} "
          f"| Ganados: {len(ganados)}")
    return detalle


def fig7_red(df: pd.DataFrame) -> None:
    """La red bipartita, para fijar el concepto."""
    try:
        import networkx as nx
    except ImportError:
        print("  ! networkx no instalado; se omite fig7")
        return

    jobs = sorted(df.job.unique())
    fig, axes = plt.subplots(1, len(jobs), figsize=(6.5 * len(jobs), 7.5),
                             squeeze=False)

    for ax, job in zip(axes[0], jobs):
        sub = df[(df.job == job) & (df.model == MODELO_FOCO)]
        G = nx.Graph()
        for _, r in sub.iterrows():
            G.add_node(r.aa_label, lado="aa", clase=r.clase)
            G.add_node(r.nt_label, lado="nt")
            G.add_edge(r.aa_label, r.nt_label, d=r.distance)

        aas = sorted([n for n, a in G.nodes(data=True) if a["lado"] == "aa"],
                     key=lambda s: int(s[3:]))
        nts = sorted([n for n, a in G.nodes(data=True) if a["lado"] == "nt"],
                     key=lambda s: (s[-1], int(s[2:-1])))
        pos = {n: (0, -i * (len(nts) / max(len(aas), 1))) for i, n in enumerate(aas)}
        pos.update({n: (1, -i * (len(aas) / max(len(nts), 1))) for i, n in enumerate(nts)})

        anchos = [max(0.3, (UMBRAL_VDW - G[u][v]["d"]) * 1.1) for u, v in G.edges()]
        nx.draw_networkx_edges(G, pos, width=anchos, edge_color="#AAAAAA",
                               alpha=0.7, ax=ax)
        nx.draw_networkx_nodes(G, pos, nodelist=aas, node_size=320, ax=ax,
                               node_color=[COLOR_CLASE[G.nodes[n]["clase"]] for n in aas])
        nx.draw_networkx_nodes(G, pos, nodelist=nts, node_size=320, ax=ax,
                               node_color="#FDBF6F", node_shape="s")
        nx.draw_networkx_labels(G, pos, font_size=6.5, ax=ax)
        ax.set_title(f"{job.capitalize()} — modelo {MODELO_FOCO}\n"
                     f"proteína (izq.) · ADN (der.)")
        ax.set_axis_off()

    fig.suptitle("Red bipartita de contactos — grosor = cercanía",
                 y=1.0, fontsize=13, fontweight="bold")
    fig.tight_layout()
    fig.savefig(FIGDIR / "fig7_red_bipartita.png")
    plt.close(fig)


def fig8_scores() -> None:
    """Resultado global por modelo."""
    try:
        s = pd.read_csv("hdock_summary.csv")
    except FileNotFoundError:
        print("  ! hdock_summary.csv no encontrado; se omite fig8")
        return

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 5))
    jobs = sorted(s.job.unique())
    alto = 0.38

    for i, job in enumerate(jobs):
        g = s[s.job == job].sort_values("model")
        y = g.model + (i - (len(jobs) - 1) / 2) * alto
        ax1.barh(y, g.docking_score, height=alto, label=job.capitalize(),
                 color=COLOR_JOB.get(job, "#777"))
        if "confidence_score" in g:
            ax2.barh(y, g.confidence_score, height=alto, label=job.capitalize(),
                     color=COLOR_JOB.get(job, "#777"))

    ax1.set_xlabel("Docking score (más negativo = mejor)")
    ax1.set_ylabel("Modelo")
    ax1.set_title("Docking score por modelo")
    ax1.invert_yaxis(); ax1.legend(frameon=False)

    ax2.axvline(0.7, color="black", ls="--", lw=1)
    ax2.text(0.705, 0.6, "  interacción\n  muy probable", fontsize=8, style="italic")
    ax2.set_xlabel("Confidence score (0–1)")
    ax2.set_title("Confianza por modelo")
    ax2.invert_yaxis(); ax2.legend(frameon=False)

    for ax in (ax1, ax2):
        ax.set_yticks(sorted(s.model.unique()))
        ax.grid(axis="x", alpha=0.25, linestyle=":")

    fig.suptitle("Resultado global del docking", y=1.02, fontsize=13,
                 fontweight="bold")
    fig.tight_layout()
    fig.savefig(FIGDIR / "fig8_scores.png")
    plt.close(fig)

In [ ]:
# ---------------------------------------------------------------------------
# 3. MAIN
# ---------------------------------------------------------------------------

def main() -> None:
    FIGDIR.mkdir(exist_ok=True)
    TABDIR.mkdir(exist_ok=True)

    print("=== Normalizando receptor/ligando ===")
    df = cargar_y_normalizar()
    df.to_csv(TABDIR / "tabla_pares_normalizada.csv", index=False,
              encoding="utf-8-sig")
    print(f"\n  tabla_pares_normalizada.csv  ({len(df)} contactos)")

    resumen_aa = (df.groupby(["job", "model", "aa_label", "clase"])
                    .agg(n_contactos=("distance", "size"),
                         dist_min=("distance", "min"))
                    .reset_index()
                    .sort_values(["job", "model", "n_contactos"],
                                 ascending=[True, True, False]))
    resumen_aa.to_csv(TABDIR / "tabla_resumen_aminoacidos.csv", index=False,
                      encoding="utf-8-sig")
    print(f"  tabla_resumen_aminoacidos.csv  ({len(resumen_aa)} filas)")

    print("\n=== Generando figuras ===")
    for nombre, fn in [
        ("fig1 mapa de contactos", lambda: fig1_mapa_contactos(df)),
        ("fig2 huella sobre el ADN", lambda: fig2_huella(df)),
        ("fig3 distribución de distancias", lambda: fig3_distancias(df)),
        ("fig4 grado por aminoácido", lambda: fig4_grado_aminoacidos(df)),
        ("fig5 heatmap aa × nt", lambda: fig5_heatmap(df)),
        ("fig6 pares comunes", lambda: fig6_pares_comunes(df)),
        ("fig7 red bipartita", lambda: fig7_red(df)),
        ("fig8 scores", fig8_scores),
    ]:
        try:
            fn()
            print(f"  OK  {nombre}")
        except Exception as exc:
            print(f"  !   {nombre} falló: {type(exc).__name__}: {exc}")

    print(f"\nFiguras en ./{FIGDIR}/   Tablas en ./{TABDIR}/")


if __name__ == "__main__":
    main()

=== Normalizando receptor/ligando ===

  Orden original de entrada en cada job:
    mutada   -> receptor = proteína
    nativa   -> receptor = proteína

  Cadenas detectadas tras normalizar:
    mutada   proteína: ['A']  ADN: ['A', 'S']  ADN pos. 2–171
    nativa   proteína: ['A']  ADN: ['A', 'S']  ADN pos. 16–173

  tabla_pares_normalizada.csv  (820 contactos)
  tabla_resumen_aminoacidos.csv  (376 filas)

=== Generando figuras ===
  OK  fig1 mapa de contactos
  OK  fig2 huella sobre el ADN
  OK  fig3 distribución de distancias
  OK  fig4 grado por aminoácido
  OK  fig5 heatmap aa × nt
  Conservados: 0 | Perdidos: 31 | Ganados: 57
  OK  fig6 pares comunes
  OK  fig7 red bipartita
  OK  fig8 scores

Figuras en ./figuras/   Tablas en ./tablas/


In [ ]:
# A) Descargar a tu computadora
from google.colab import files
# files.download('hdock_pairs.csv')

# B) Bajar todo de un golpe, comprimido
!zip -r resultados.zip tablas figuras *.csv
files.download('resultados.zip')

# C) Lo mejor para la clase: montar Drive y escribir ahí
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p "/content/drive/MyDrive/practica6"
# !cp -r *.csv tablas figuras "/content/drive/MyDrive/practica6/"

  adding: tablas/ (stored 0%)
  adding: tablas/tabla_resumen_aminoacidos.csv (deflated 79%)
  adding: tablas/tabla_pares_comunes.csv (deflated 74%)
  adding: tablas/tabla_pares_normalizada.csv (deflated 77%)
  adding: figuras/ (stored 0%)
  adding: figuras/fig2_huella_adn.png (deflated 23%)
  adding: figuras/fig4_grado_aminoacidos.png (deflated 24%)
  adding: figuras/fig3_distancias.png (deflated 14%)
  adding: figuras/fig6_pares_comunes.png (deflated 21%)
  adding: figuras/fig8_scores.png (deflated 26%)
  adding: figuras/fig5_heatmap.png (deflated 16%)
  adding: figuras/fig1_contactos_modelo1.png (deflated 15%)
  adding: figuras/fig7_red_bipartita.png (deflated 8%)
  adding: hdock_ligand_residues.csv (deflated 70%)
  adding: hdock_matriz_pares.csv (deflated 79%)
  adding: hdock_pairs.csv (deflated 75%)
  adding: hdock_receptor_residues.csv (deflated 78%)
  adding: hdock_summary.csv (deflated 53%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>